# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.8 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID='task379'; CH=10; N=30
TASK_JSON=Path(COMPETITION)/'task379.json'

OUT_DIR=Path.cwd()/'task379_v12_priority_adversarial_onnx'; 
OUT_DIR.mkdir(exist_ok=True)
ONNX_PATH=OUT_DIR/f'{TASK_ID}.onnx'; 
SUMMARY_PATH=OUT_DIR/f'{TASK_ID}_validation_summary.json'

SUBMISSION=Path.cwd()/'submission.zip'

with TASK_JSON.open() as f: task=json.load(f)

In [6]:
def grid_to_tensor(grid):
    a=np.asarray(grid,dtype=np.int64); x=np.zeros((1,CH,N,N),np.float32)
    h,w=a.shape
    for r in range(h):
        for c in range(w): x[0,int(a[r,c]),r,c]=1.0
    return x

class Task379PriorityAdversarial(nn.Module):
    def __init__(self):
        super().__init__()
        idx=torch.arange(N,dtype=torch.float32)
        q=idx.view(1,1,N,1)
        r=idx.view(1,1,1,N)
        self.register_buffer('le',(r<=q).float()) # output q >= marker r
        self.register_buffer('ge',(r>=q).float()) # output q <= marker r
        self.register_buffer('lt',(r<q).float())  # output q > marker r
        self.register_buffer('gt',(r>q).float())  # output q < marker r
        rr=idx.view(1,1,N,1).expand(1,1,N,N)
        cc=idx.view(1,1,1,N).expand(1,1,N,N)
        self.register_buffer('priority',(rr*N+cc+1.0)) # [1,1,H,W]
    def shift_next_1d(self,z):
        return torch.cat([z[:,:,1:], z[:,:,:1]*0.0], dim=2)
    def shift_prev_1d(self,z):
        return torch.cat([z[:,:,:1]*0.0, z[:,:,:-1]], dim=2)
    def shift2(self,z,dr:int,dc:int):
        if dr==1:
            z=torch.cat([z[:,:,:1,:]*0.0, z[:,:,:N-1,:]], dim=2)
        elif dr==-1:
            z=torch.cat([z[:,:,1:,:], z[:,:,:1,:]*0.0], dim=2)
        if dc==1:
            z=torch.cat([z[:,:,:,:1]*0.0, z[:,:,:,:N-1]], dim=3)
        elif dc==-1:
            z=torch.cat([z[:,:,:,1:], z[:,:,:,:1]*0.0], dim=3)
        return z
    def dilate3_shift(self,z):
        acc=z*0.0
        for dr in (-1,0,1):
            for dc in (-1,0,1):
                acc=torch.clamp(acc+self.shift2(z,dr,dc),0.0,1.0)
        return acc
    def reduce_sum_marker(self, rel, marker_priority):
        # rel: [B,1,N,N,1], marker_priority: [B,9,1,N,M]
        # Sum is faster/export-stable; later we combine h/v/original by elementwise max to avoid duplicate path inflation.
        return (rel*marker_priority).sum(dim=3).clamp(0.0, 901.0)
    def solve_axis(self, marker_ch, priority_ch, bar_any):
        # marker_ch/priority_ch: [B,9,N,M], bar_any: [B,1,N]
        cum=torch.cumsum(bar_any, dim=2)
        total=cum[:,:,-1:]
        cum_q=cum.unsqueeze(3) # [B,1,q,1]
        cum_r=cum.unsqueeze(2) # [B,1,1,r]
        same_segment=(torch.abs(cum_q-cum_r)<0.5).float()
        below_exists=(total-cum>0.5).float().unsqueeze(3)
        above_exists=(cum>0.5).float().unsqueeze(3)
        next_not_bar=(1.0-self.shift_next_1d(bar_any)).unsqueeze(3)
        prev_not_bar=(1.0-self.shift_prev_1d(bar_any)).unsqueeze(3)
        nonbar=(1.0-bar_any).unsqueeze(3)
        marker=marker_ch.unsqueeze(2)
        pr=priority_ch.unsqueeze(2)
        # bridge positions, with priority propagated from the winning source marker.
        bridge_down_rel=self.le.unsqueeze(-1) * same_segment.unsqueeze(-1) * below_exists.unsqueeze(-1) * next_not_bar.unsqueeze(-1) * nonbar.unsqueeze(-1)
        bridge_up_rel=self.ge.unsqueeze(-1) * same_segment.unsqueeze(-1) * above_exists.unsqueeze(-1) * prev_not_bar.unsqueeze(-1) * nonbar.unsqueeze(-1)
        bridge_score=torch.maximum(self.reduce_sum_marker(bridge_down_rel, pr), self.reduce_sum_marker(bridge_up_rel, pr))
        # center at nearest rail line below/above marker.
        barq=bar_any.unsqueeze(3).unsqueeze(-1)
        center_down_rel=self.lt.unsqueeze(-1) * (torch.abs(cum_r-(cum_q-1.0))<0.5).float().unsqueeze(-1) * barq
        center_up_rel=self.gt.unsqueeze(-1) * (torch.abs(cum_r-cum_q)<0.5).float().unsqueeze(-1) * barq
        center_score=torch.maximum(self.reduce_sum_marker(center_down_rel, pr), self.reduce_sum_marker(center_up_rel, pr))
        center=(center_score>0.5).float()
        bridge=(bridge_score>0.5).float()
        return center, bridge, center_score, bridge_score
    def forward(self,x):
        active=(x.sum(dim=1,keepdim=True)>0.5).float()
        colors=x[:,1:10]*active
        nonzero=colors.sum(dim=1,keepdim=True)
        row_width=active.sum(dim=3,keepdim=True)
        col_height=active.sum(dim=2,keepdim=True)
        row_color_counts=colors.sum(dim=3,keepdim=True)
        col_color_counts=colors.sum(dim=2,keepdim=True)
        # Exact full same-color reference axes over raw active canvas.
        hbar_ch=((row_color_counts>=row_width-0.5)&(row_width>0.5)).float()
        vbar_ch=((col_color_counts>=col_height-0.5)&(col_height>0.5)).float()
        # Keep only the rail color: the color with most full rows/cols.
        score=hbar_ch.squeeze(3).sum(dim=2)+vbar_ch.squeeze(2).sum(dim=2) # [B,9]
        rail_idx=torch.argmax(score,dim=1)
        rail_oh=torch.nn.functional.one_hot(rail_idx,num_classes=9).float().view(1,9,1,1)
        hbar_ch=hbar_ch*rail_oh
        vbar_ch=vbar_ch*rail_oh
        hrail_ch=hbar_ch.expand(-1,-1,-1,N)*active
        vrail_ch=vbar_ch.expand(-1,-1,N,-1)*active
        rail_ch=torch.clamp(hrail_ch+vrail_ch,0.0,1.0)
        rail_any=torch.clamp(rail_ch.sum(dim=1,keepdim=True),0.0,1.0)
        markers=colors*(1.0-rail_any)
        priority_ch=markers*self.priority
        hbar_any=torch.clamp(hbar_ch.sum(dim=1,keepdim=False),0.0,1.0).transpose(1,2) # [B,1,H]
        vbar_any=torch.clamp(vbar_ch.sum(dim=1,keepdim=False),0.0,1.0)              # [B,1,W]
        center_h, bridge_h, score_center_h, score_bridge_h = self.solve_axis(markers, priority_ch, hbar_any)
        center_v_t, bridge_v_t, score_center_v_t, score_bridge_v_t = self.solve_axis(markers.transpose(2,3), priority_ch.transpose(2,3), vbar_any)
        center_v=center_v_t.transpose(2,3); bridge_v=bridge_v_t.transpose(2,3)
        score_center_v=score_center_v_t.transpose(2,3); score_bridge_v=score_bridge_v_t.transpose(2,3)
        centers=torch.clamp(center_h+center_v,0.0,1.0)*active
        # Priority scores for generated marker writes.
        marker_score=torch.maximum(priority_ch, torch.maximum(score_center_h, score_bridge_h))
        marker_score=torch.maximum(marker_score, torch.maximum(score_center_v, score_bridge_v))*active
        winner_idx=torch.argmax(marker_score, dim=1)  # [B,H,W]
        winner_oh=torch.nn.functional.one_hot(winner_idx, num_classes=9).float().permute(0,3,1,2)
        marker_any=(marker_score.sum(dim=1,keepdim=True)>0.5).float()*active
        marker_final=winner_oh*marker_any
        center_any_h=torch.clamp(center_h.sum(dim=1,keepdim=True),0.0,1.0)
        center_any_v=torch.clamp(center_v.sum(dim=1,keepdim=True),0.0,1.0)
        cap_ch=torch.clamp(self.dilate3_shift(center_any_h*hrail_ch)+self.dilate3_shift(center_any_v*vrail_ch),0.0,1.0)*active
        rail_out=torch.clamp(rail_ch+cap_ch,0.0,1.0)*active*(1.0-marker_any)
        # Preserve any non-axis colored content not generated over. Usually this is empty because markers are marker_final.
        occupied=torch.clamp(rail_out.sum(dim=1,keepdim=True)+marker_any,0.0,1.0)
        preserved=colors*(1.0-rail_any)*(1.0-occupied)*active
        out_col=torch.clamp(preserved+rail_out+marker_final,0.0,1.0)*active
        occ=torch.clamp(out_col.sum(dim=1,keepdim=True),0.0,1.0)
        out0=active*(1.0-occ)
        return torch.cat([out0,out_col],dim=1)*active

def validate_ort(sess, examples):
    inp=sess.get_inputs()[0].name
    ok=0; bad=[]; diffs=[]; outside=0; cover=0; onehot=0
    for i,ex in enumerate(examples):
        y=sess.run(None,{inp:grid_to_tensor(ex['input'])})[0]
        exp=grid_to_tensor(ex['output'])
        eq=np.array_equal(y,exp); ok+=int(eq)
        if not eq:
            bad.append(i); diffs.append(int(np.abs(y-exp).sum()))
        h,w=np.asarray(ex['output']).shape
        outside+=int((np.abs(y[:,:,h:,:]).sum()+np.abs(y[:,:,:,w:]).sum())==0)
        cover+=int(np.allclose(y[0,:,:h,:w].sum(axis=0),1.0))
        onehot+=int(np.max(y[0,:,:h,:w].sum(axis=0))<=1.0001)
    return {'ok':ok,'total':len(examples),'bad_first10':bad[:10],'diffs_first10':diffs[:10],'outside_zero_ok':outside,'active_canvas_covered_ok':cover,'onehot_ok':onehot}


In [7]:
model = Task379PriorityAdversarial().eval()


# Fast PyTorch sanity on visible data.
with torch.no_grad():
    for name, examples in [('train', task['train']), ('test', task['test'])]:
        ok = 0
        for ex in examples:
            y = model(torch.from_numpy(grid_to_tensor(ex['input']))).numpy()
            ok += int(np.array_equal(y, grid_to_tensor(ex['output'])))
        print('torch', name, ok, '/', len(examples))

dummy = torch.from_numpy(grid_to_tensor(task['test'][0]['input']))
# This graph uses per-marker priority resolution. In this environment the torch dynamo exporter is much faster than the legacy exporter.
try:
    import onnxscript  # required by torch.onnx.export(..., dynamo=True)
except Exception as e:
    raise RuntimeError('onnxscript is required for dynamo=True ONNX export. Install onnxscript or use the prebuilt ONNX artifact.') from e

torch.onnx.export(
    model, dummy, str(ONNX_PATH),
    input_names=['input'], output_names=['output'],
    opset_version=18, dynamo=True, external_data=False,
)
print('exported', ONNX_PATH, ONNX_PATH.stat().st_size)

torch train 3 / 3
torch test 1 / 1
[torch.onnx] Obtain model graph for `Task379PriorityAdversarial()` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Task379PriorityAdversarial()` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
exported /kaggle/working/task379_v12_priority_adversarial_onnx/task379.onnx 244147


In [8]:
# ONNXRuntime validation and submission generation

sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0].name

def validate(examples):
    ok = 0; bad = []; outside = 0; cover = 0; onehot = 0
    t = time.time()
    for i, ex in enumerate(examples):
        y = sess.run(None, {inp: grid_to_tensor(ex['input'])})[0]
        exp = grid_to_tensor(ex['output'])
        eq = np.array_equal(y, exp)
        ok += int(eq)
        if not eq and len(bad) < 10:
            bad.append({'i': i, 'id': ex.get('id',''), 'purpose': ex.get('purpose',''), 'diff': int(np.abs(y-exp).sum())})
        h, w = np.asarray(ex['output']).shape
        outside += int((np.abs(y[:,:,h:,:]).sum() + np.abs(y[:,:,:,w:]).sum()) == 0)
        cover += int(np.allclose(y[0,:,:h,:w].sum(axis=0), 1.0))
        onehot += int(np.max(y[0,:,:h,:w].sum(axis=0)) <= 1.0001)
    return {'ok': ok, 'total': len(examples), 'bad_first10': bad,
            'outside_zero_ok': outside, 'active_canvas_covered_ok': cover,
            'onehot_ok': onehot, 'seconds': round(time.time()-t, 3)}



In [9]:
# Validate arc-gen in two chunks to keep notebook progress visible.
res_train = validate(task['train'])
res_test = validate(task['test'])
res_arc60 = validate(task['arc-gen'][:157])
res_arc_tail = validate(task['arc-gen'][157:])

onnx_model = onnx.load(str(ONNX_PATH), load_external_data=False)
ops = collections.Counter(n.op_type for n in onnx_model.graph.node)
def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]
summary = {
    'model_family': 'task379_v12_priority_adversarial_cumsum_coordinate_projector',
    'onnxruntime_version': ort.__version__,
    'input_shape': vi_shape(onnx_model.graph.input[0]),
    'output_shape': vi_shape(onnx_model.graph.output[0]),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'ops': dict(ops),
    'forbidden_ops': sorted(set(ops) & {'Loop','Scan','NonZero','Unique','Script','Function'}),
    'function_count': len(onnx_model.functions),
    'uses_cumsum': 'CumSum' in ops,
    'uses_maxpool': 'MaxPool' in ops,
    'uses_pad': 'Pad' in ops,
    'uses_einsum': 'Einsum' in ops,
    'train': res_train,
    'test': res_test,
    'arc_gen_60pct_holdout': res_arc60,
    'arc_gen_tail_105': res_arc_tail,
    'arc_gen_all': {'ok': res_arc60['ok'] + res_arc_tail['ok'], 'total': res_arc60['total'] + res_arc_tail['total']},

}
json.dump(summary, open(SUMMARY_PATH, 'w'), indent=2)
print(json.dumps({k:v for k,v in summary.items() if k != 'ops'}, indent=2))
assert summary['train']['ok'] == summary['train']['total']
assert summary['test']['ok'] == summary['test']['total']
assert summary['arc_gen_all']['ok'] == summary['arc_gen_all']['total']

assert summary['onnx_size_bytes'] < 1_400_000
assert not summary['forbidden_ops'] and summary['function_count'] == 0

{
  "model_family": "task379_v12_priority_adversarial_cumsum_coordinate_projector",
  "onnxruntime_version": "1.27.0",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 244147,
  "forbidden_ops": [],
  "function_count": 0,
  "uses_cumsum": true,
  "uses_maxpool": false,
  "uses_pad": false,
  "uses_einsum": false,
  "train": {
    "ok": 3,
    "total": 3,
    "bad_first10": [],
    "outside_zero_ok": 3,
    "active_canvas_covered_ok": 3,
    "onehot_ok": 3,
    "seconds": 0.015
  },
  "test": {
    "ok": 1,
    "total": 1,
    "bad_first10": [],
    "outside_zero_ok": 1,
    "active_canvas_covered_ok": 1,
    "onehot_ok": 1,
    "seconds": 0.004
  },
  "arc_gen_60pct_holdout": {
    "ok": 157,
    "total": 157,
    "bad_first10": [],
    "outside_zero_ok": 157,
    "active_canvas_covered_ok": 157,
    "onehot_ok": 157,
    "seconds": 0.568
  },
  "arc_gen_tail_105": {
    "ok": 105,
    "total": 105,
    "b

In [10]:
for zp in [SUBMISSION]:
    if zp.exists(): zp.unlink()
    with zipfile.ZipFile(zp, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
    print('wrote', zp, zipfile.ZipFile(zp).namelist())

wrote /kaggle/working/submission.zip ['task379.onnx']
